In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
patientpayer_table = dbutils.widgets.get("patientpayer_table")
patient_table = dbutils.widgets.get("patient_table")
branch_table = dbutils.widgets.get("branch_table")
payer_table = dbutils.widgets.get("payer_table")
person_table = dbutils.widgets.get("person_table")
payeraddress_table = dbutils.widgets.get("payeraddress_table")
address_table = dbutils.widgets.get("address_table")
payerphonenumber_table = dbutils.widgets.get("payerphonenumber_table")
phonenumber_table = dbutils.widgets.get("phonenumber_table")
personaddress_table = dbutils.widgets.get("personaddress_table")
personphonenumber_table = dbutils.widgets.get("personphonenumber_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW insurance_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS STRING) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(InsRank AS INT) AS InsRank,
  CAST(ActiveIns AS INT) AS ActiveIns,
  CAST(InsCode AS STRING) AS InsCode,
  NULL AS InsEDIID,
  NULL AS InsEID,
  CAST(InsGroupID AS STRING) AS InsGroupID,
  NULL AS InsPolicy,
  NULL AS InsPreCertNbr,
  CAST(InsBalance AS DOUBLE) AS InsBalance,
  CAST(InsPymt AS DOUBLE) AS InsPymt,
  CAST(InsAdj AS DOUBLE) AS InsAdj,
  CAST(InsName AS STRING) AS InsName,
  CAST(InsAddr1 AS STRING) AS InsAddr1,
  CAST(InsAddr2 AS STRING)  AS InsAddr2, 
  CAST(InsCity AS STRING) AS InsCity,
  CAST(InsState AS STRING) AS InsState,
  CAST(InsZip AS STRING) AS InsZip,
  NULL AS InsCountry,
  CAST(InsProvince AS STRING) AS InsProvince,
  CAST(InsPhone AS STRING) AS InsPhone,
  CAST(InsEmail AS STRING) AS InsEmail,
  CAST(InsSubFName AS STRING) AS InsSubFName,
  CAST(InsSubMName AS STRING) AS InsSubMName,
  CAST(InsSubLName AS STRING) AS InsSubLName,
  NULL AS InsSubSuffix,
  CAST(InsSubAddr1 AS STRING) AS InsSubAddr1,
  CAST(InsSubAddr2 AS STRING) AS InsSubAddr2,
  CAST(InsSubCity AS STRING) AS InsSubCity,
  CAST(InsSubState AS STRING) AS InsSubState,
  CAST(InsSubZip AS STRING) AS InsSubZip,
  NULL AS InsSubCountry,
  NULL AS InsSubProvince,
  CAST(InsSubPhone AS STRING) AS InsSubPhone,
  CAST(InsSubDOB AS STRING) AS InsSubDOB,
  NULL AS InsSubSSN,
  CAST(InsSubGender AS STRING) AS InsSubGender,
  NULL AS InsSubRelation,
  CAST(InitialBillDate AS STRING) AS InitialBillDate,
  NULL AS LastBillDate,
  NULL AS LastBillSubmitDate,
  NULL AS LastBillType,
  NULL AS LastMediaType,
  NULL AS InsStatusCode,
  NULL AS InsStatusDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  insurance_cte AS (
    SELECT  
    CAST('{fetch_date}' AS DATE) AS ReportingDate,
    b.ExternalId AS FacilityCode,
    bl.ClaimNumber AS AcctNbr,
    '1' AS InsRank,
    '1' AS ActiveIns,
    pay.Id AS InsCode,
    -- pay.EdiId AS InsEDIID,
    pp.GroupNumber AS InsGroupID,
    -- pp.PayerIdNumber AS InsPolicy,
    bl.AmountBilled AS InsBalance,
    0.00 AS InsPymt,
    0.00 AS InsAdj,
    CASE WHEN pay.Name LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(pay.Name, '"', ' '))) ELSE pay.Name END AS InsName,
    CASE WHEN payAddr.Street1 LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(payAddr.Street1, '"', ' '))) ELSE payAddr.Street1 END AS InsAddr1,
    CASE WHEN payAddr.Street2 LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(payAddr.Street2, '"', ' '))) ELSE payAddr.Street2 END AS InsAddr2,
    payAddr.City AS InsCity,
    payAddr.State AS InsState,
    payAddr.ZipCode AS InsZip,
    p.MedicaidProgram AS InsProvince,
    -- payPhone.Number AS InsPhone,
    'NOT SELECTED' AS InsPhone,
    per.Email AS InsEmail,
    CASE WHEN per.FirstName LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(per.FirstName, '"', ' '))) ELSE per.FirstName END AS InsSubFName,
    CASE WHEN per.MiddleName LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(per.MiddleName, '"', ' '))) ELSE per.MiddleName END AS InsSubMName,
    CASE WHEN per.LastName LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(per.LastName, '"', ' '))) ELSE per.LastName END AS InsSubLName,
    CASE WHEN perAddr.Street1 LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(perAddr.Street1, '"', ' '))) ELSE perAddr.Street1 END AS InsSubAddr1,
    CASE WHEN perAddr.Street2 LIKE '%"%' THEN LTRIM(RTRIM(REPLACE(perAddr.Street2, '"', ' '))) ELSE perAddr.Street2 END AS InsSubAddr2,
    perAddr.City AS InsSubCity,
    perAddr.State AS InsSubState,
    perAddr.ZipCode AS InsSubZip,
    CONCAT('(', SUBSTRING(perPhone.Number, 1, 3), ') ', SUBSTRING(perPhone.Number, 4, 3), '-', SUBSTRING(perPhone.Number, 7, 4)) AS InsSubPhone,
    DATE_FORMAT(per.DateOfBirth, 'yyyyMMdd') AS InsSubDOB,
    CASE per.Gender 
        WHEN 1 THEN 'M'
        WHEN 2 THEN 'F'
        ELSE 'U'
    END AS InsSubGender,
    DATE_FORMAT(bl.DateBilled, 'yyyy-MM-dd') AS InitialBillDate,
    '19' AS SourceSystemKey
  FROM {source_table} bl
  JOIN {patientpayer_table} pp ON pp.Id = bl.PatientPayerId
  JOIN {patient_table} p ON p.Id = pp.PatientId
  JOIN {branch_table} b ON b.Id = p.BranchId
  JOIN {payer_table} pay ON pay.Id = pp.PayerId
  JOIN {person_table} per ON per.Id = p.PersonId
  -- -- uncomment this rn logics to avoid explolosion on payer details
  -- -- Payer address (pick first one)
  LEFT JOIN (
      SELECT pa.PayerId, a.Street1, a.Street2, a.City, a.State, a.ZipCode,
            ROW_NUMBER() OVER (PARTITION BY pa.PayerId ORDER BY pa.Id) as rn
      FROM {payeraddress_table} pa
      JOIN {address_table} a ON a.Id = pa.AddressId
  ) payAddr ON payAddr.PayerId = pay.Id AND payAddr.rn = 1
  -- Payer phone (pick first one)
  LEFT JOIN (
      SELECT ppn.PayerId, pn.Number,
            ROW_NUMBER() OVER (PARTITION BY ppn.PayerId ORDER BY ppn.Id) as rn
      FROM {payerphonenumber_table} ppn
      JOIN {phonenumber_table} pn ON pn.Id = ppn.PhoneId
  ) payPhone ON payPhone.PayerId = pay.Id AND payPhone.rn = 1
  -- Person (subscriber) address (pick first one)
  LEFT JOIN (
      SELECT perAddrLink.PersonId, a.Street1, a.Street2, a.City, a.State, a.ZipCode,
            ROW_NUMBER() OVER (PARTITION BY perAddrLink.PersonId ORDER BY perAddrLink.Id) as rn
      FROM {personaddress_table} perAddrLink
      JOIN {address_table} a ON a.Id = perAddrLink.AddressId
  ) perAddr ON perAddr.PersonId = per.Id AND perAddr.rn = 1
  -- Person (subscriber) phone (pick first one)
  LEFT JOIN (
      SELECT perPhoneLink.PersonId, pn.Number,
            ROW_NUMBER() OVER (PARTITION BY perPhoneLink.PersonId ORDER BY perPhoneLink.Id) as rn
      FROM {personphonenumber_table} perPhoneLink
      JOIN {phonenumber_table} pn ON pn.Id = perPhoneLink.PhoneId
  ) perPhone ON perPhone.PersonId = per.Id AND perPhone.rn = 1
  -- WHERE bl.ClaimNumber IN ('337128FJ1115', '327691FJ1562', '327682FJ1507')

  -- -- Payer address
  -- LEFT JOIN prd_bronze_raw.cubhub.payeraddress pa ON pa.PayerId = pay.Id
  -- LEFT JOIN prd_bronze_raw.cubhub.address payAddr ON payAddr.Id = pa.AddressId
  -- -- Payer phone
  -- LEFT JOIN prd_bronze_raw.cubhub.payerphonenumber ppn ON ppn.PayerId = pay.Id
  -- LEFT JOIN prd_bronze_raw.cubhub.phonenumber payPhone ON payPhone.Id = ppn.PhoneId
  -- -- Person (subscriber) address
  -- LEFT JOIN prd_bronze_raw.cubhub.personaddress perAddrLink ON perAddrLink.PersonId = per.Id
  -- LEFT JOIN prd_bronze_raw.cubhub.address perAddr ON perAddr.Id = perAddrLink.AddressId
  -- -- Person (subscriber) phone
  -- LEFT JOIN prd_bronze_raw.cubhub.personphonenumber perPhoneLink ON perPhoneLink.PersonId = per.Id
  -- LEFT JOIN prd_bronze_raw.cubhub.phonenumber perPhone ON perPhone.Id = perPhoneLink.PhoneId
  -- WHERE bl.ClaimNumber IN ('337128FJ1115', '327691FJ1562', '327682FJ1507')

  WHERE bl.isActive='true'
  ),
  insurance_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM insurance_cte
  )
  SELECT 
    ReportingDate,
    FacilityCode,
    AcctNbr,
    InsRank,
    ActiveIns,
    InsCode,
    InsGroupID,
    InsBalance,
    InsPymt,
    InsAdj,
    InsName,
    InsAddr1,
    InsAddr2,
    InsCity,
    InsState,
    InsZip,
    InsProvince,
    InsPhone,
    InsEmail,
    InsSubFName,
    InsSubMName,
    InsSubLName,
    InsSubAddr1,
    InsSubAddr2,
    InsSubCity,
    InsSubState,
    InsSubZip,
    InsSubPhone,
    InsSubDOB,
    InsSubGender,
    InitialBillDate,
    SourceSystemKey
  FROM insurance_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING insurance_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 19

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.InsRank = src.InsRank,
    tgt.ActiveIns = src.ActiveIns,
    tgt.InsCode = src.InsCode,
    tgt.InsEDIID = src.InsEDIID,
    tgt.InsEID = src.InsEID,
    tgt.InsGroupID = src.InsGroupID,
    tgt.InsPolicy = src.InsPolicy,
    tgt.InsPreCertNbr = src.InsPreCertNbr,
    tgt.InsBalance = src.InsBalance,
    tgt.InsPymt = src.InsPymt,
    tgt.InsAdj = src.InsAdj,
    tgt.InsName = src.InsName,
    tgt.InsAddr1 = src.InsAddr1,
    tgt.InsAddr2 = src.InsAddr2,
    tgt.InsCity = src.InsCity,
    tgt.InsState = src.InsState,
    tgt.InsZip = src.InsZip,
    tgt.InsCountry = src.InsCountry,
    tgt.InsProvince = src.InsProvince,
    tgt.InsPhone = src.InsPhone,
    tgt.InsEmail = src.InsEmail,
    tgt.InsSubFName = src.InsSubFName,
    tgt.InsSubMName = src.InsSubMName,
    tgt.InsSubLName = src.InsSubLName,
    tgt.InsSubSuffix = src.InsSubSuffix,
    tgt.InsSubAddr1 = src.InsSubAddr1,
    tgt.InsSubAddr2 = src.InsSubAddr2,
    tgt.InsSubCity = src.InsSubCity,
    tgt.InsSubState = src.InsSubState,
    tgt.InsSubZip = src.InsSubZip,
    tgt.InsSubCountry = src.InsSubCountry,
    tgt.InsSubProvince = src.InsSubProvince,
    tgt.InsSubPhone = src.InsSubPhone,
    tgt.InsSubDOB = src.InsSubDOB,
    tgt.InsSubSSN = src.InsSubSSN,
    tgt.InsSubGender = src.InsSubGender,
    tgt.InsSubRelation = src.InsSubRelation,
    tgt.InitialBillDate = src.InitialBillDate,
    tgt.LastBillDate = src.LastBillDate,
    tgt.LastBillSubmitDate = src.LastBillSubmitDate,
    tgt.LastBillType = src.LastBillType,
    tgt.LastMediaType = src.LastMediaType,
    tgt.InsStatusCode = src.InsStatusCode,
    tgt.InsStatusDate = src.InsStatusDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    InsRank,
    ActiveIns,
    InsCode,
    InsEDIID,
    InsEID,
    InsGroupID,
    InsPolicy,
    InsPreCertNbr,
    InsBalance,
    InsPymt,
    InsAdj,
    InsName,
    InsAddr1,
    InsAddr2,
    InsCity,
    InsState,
    InsZip,
    InsCountry,
    InsProvince,
    InsPhone,
    InsEmail,
    InsSubFName,
    InsSubMName,
    InsSubLName,
    InsSubSuffix,
    InsSubAddr1,
    InsSubAddr2,
    InsSubCity,
    InsSubState,
    InsSubZip,
    InsSubCountry,
    InsSubProvince,
    InsSubPhone,
    InsSubDOB,
    InsSubSSN,
    InsSubGender,
    InsSubRelation,
    InitialBillDate,
    LastBillDate,
    LastBillSubmitDate,
    LastBillType,
    LastMediaType,
    InsStatusCode,
    InsStatusDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.InsRank,
    src.ActiveIns,
    src.InsCode,
    src.InsEDIID,
    src.InsEID,
    src.InsGroupID,
    src.InsPolicy,
    src.InsPreCertNbr,
    src.InsBalance,
    src.InsPymt,
    src.InsAdj,
    src.InsName,
    src.InsAddr1,
    src.InsAddr2,
    src.InsCity,
    src.InsState,
    src.InsZip,
    src.InsCountry,
    src.InsProvince,
    src.InsPhone,
    src.InsEmail,
    src.InsSubFName,
    src.InsSubMName,
    src.InsSubLName,
    src.InsSubSuffix,
    src.InsSubAddr1,
    src.InsSubAddr2,
    src.InsSubCity,
    src.InsSubState,
    src.InsSubZip,
    src.InsSubCountry,
    src.InsSubProvince,
    src.InsSubPhone,
    src.InsSubDOB,
    src.InsSubSSN,
    src.InsSubGender,
    src.InsSubRelation,
    src.InitialBillDate,
    src.LastBillDate,
    src.LastBillSubmitDate,
    src.LastBillType,
    src.LastMediaType,
    src.InsStatusCode,
    src.InsStatusDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)